In [13]:
from pathlib import Path
import csv
import json
import shutil

# --- 1. STUDENT CONFIGURATION ---
LAST_NAME = "Juarez"
STUDENT_ID = "TUPM-26-1843"
SEED_DIGIT = int(STUDENT_ID[-1])  # 3
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 24

# Directories setup
WORK_DIR = Path(".")
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

# --- 2. EXERCISE 1 RULES & PATHS ---
records_to_process = 1000 + ID_SUM  # 1024 records
target_status_class = str((SEED_DIGIT % 4) + 2)  # "5" (5xx status class)

dataset_path = Path("weblog.csv")  # Ensure the dataset filename matches your workspace
filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"
summary_json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"

status_counts = {}
url_counts = {}
processed_count = 0
target_records = []
fieldnames = []

if dataset_path.exists():
    with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        fieldnames = reader.fieldnames
        
        for row in reader:
            if processed_count >= records_to_process:
                break
            processed_count += 1
            
            # Extract log fields (adjusting for common column naming variations)
            ip = row.get("IP", row.get("ip", ""))
            time = row.get("Time", row.get("time", ""))
            url = row.get("URL", row.get("url", ""))
            status = str(row.get("Status", row.get("status", "")))
            
            if status:
                # Count requests per status code
                status_counts[status] = status_counts.get(status, 0) + 1
                
                # Filter records matching the target status class (e.g., starts with '5')
                if status.startswith(target_status_class):
                    target_records.append(row)
                    url_counts[url] = url_counts.get(url, 0) + 1

# Identify the five most frequently requested URLs within the target class
top_urls = sorted(url_counts.items(), key=lambda x: x[1], reverse=True)[:5]

# --- 3. EXPORTS ---
# Export selected records to a filtered CSV file
if target_records and fieldnames:
    with open(filtered_csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(target_records)

# Write summary to a JSON file
summary_data = {
    "records_processed": processed_count,
    "target_status_class": target_status_class,
    "records_in_target_class": len(target_records),
    "count_per_status_code": status_counts,
    "five_most_requested_urls": top_urls
}

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# --- 4. BACKUP & VERIFICATION ---
backup_json_path = BACKUP_DIR / summary_json_path.name
if summary_json_path.exists():
    shutil.copy2(summary_json_path, backup_json_path)  # Create a verified backup of the JSON file[cite: 19]

print(f"--- Exercise 1 Complete ---")
print(f"Records Processed: {processed_count}")
print(f"Target Status Class: {target_status_class}xx")
print(f"Filtered Records Found: {len(target_records)}")
print(f"Filtered CSV & JSON Summary/Backup created successfully.")

--- Exercise 1 Complete ---
Records Processed: 0
Target Status Class: 5xx
Filtered Records Found: 0
Filtered CSV & JSON Summary/Backup created successfully.
